**Euler's Gamma function - extending factorial to real numbers.**
The factorial $n!$ is defined for non-negative integers:
$0! = 1$, $1! = 1$, $2! = 2$, $3! = 6$, $\ldots$
Euler's **Gamma function** extends this to all positive real numbers
(and beyond) via an integral:

$$
\Gamma(s) = \int_0^{\infty} x^{s-1}\,e^{-x}\,dx
$$

The key property is $\Gamma(n+1) = n!$ for any non-negative integer $n$,
so $\Gamma$ is a smooth curve that passes exactly through every integer
factorial value.
This notebook computes $\Gamma$ numerically using **Simpson's rule** and
compares it against the recursive integer factorial.

In [ ]:
"""euler_gamma.ipynb"""

# Cell 01 - Integer factorial via recursion

from IPython.display import Math, display


def factorial_recursive(n: int) -> int:
    """Return n! recursively. Base case: 0! = 1."""
    if n == 0:
        return 1
    return n * factorial_recursive(n - 1)


display(Math(f"5! = {factorial_recursive(5):,}"))

**Numerically integrating the Gamma function.**
The integrand $f(x, s) = x^{s-1} e^{-x}$ decays rapidly for large $x$
(the $e^{-x}$ term dominates), so integrating to 1000 instead of
$\infty$ introduces negligible error.

Simpson's rule approximates the integral of $f$ over $[a, b]$ divided
into $N$ equal intervals of width $\Delta x = (b-a)/N$:

$$
\int_a^b f(x)\,dx \approx
\frac{\Delta x}{3}\bigl[f_0 + 4f_1 + 2f_2 + 4f_3 + \cdots + 4f_{N-1} + f_N\bigr]
$$

The alternating weights 4, 2, 4, 2, $\ldots$ arise from fitting a
quadratic through each successive triplet of points.
The weight pattern `2 * (i % 2 + 1)` in the loop produces 4 for odd
$i$ and 2 for even $i$, implementing exactly this scheme.

At the lower limit $x = 0$, the integrand is $0^{s-1} e^0$.
For $s \geq 1$ this is well-behaved, but for $s < 1$ it would be
$0^{\text{negative}}$ which NumPy returns as `inf` rather than raising
an exception.
Rather than cleaning up afterward with `np.nan_to_num`, the integral
starts at `np.finfo(float).tiny` - the smallest normalized positive
`float64`, approximately $2.2 \times 10^{-308}$, exactly one machine
bit above zero.
This sidesteps the singularity entirely while introducing no measurable
integration error.

In [ ]:
# Cell 02 - Gamma function via Simpson's rule numerical integration

import numpy as np


def f(x: float, s: float) -> float:
    """Gamma function integrand: x^(s-1) * exp(-x)."""
    return np.power(x, s - 1) * np.exp(-x)


def simpsons_rule(func, s: float, a: float, b: float, intervals: int) -> float:
    """Approximate the integral of func(x, s) over [a, b] using Simpson's rule."""
    dx = (b - a) / intervals
    area = func(a, s) + func(b, s)
    for i in range(1, intervals):
        area += func(a + i * dx, s) * (2 * (i % 2 + 1))  # weights: 4, 2, 4, 2, ...
    return dx / 3 * area


def euler_gamma(s: float) -> float:
    """Return Gamma(s) numerically. Upper limit 1000 is sufficient since
    exp(-x) makes the integrand negligible well before x=1000.
    """
    # np.finfo(float).tiny is the smallest normalized positive float64 (~2.2e-308),
    # exactly one machine bit above zero. Starting the integral here instead of
    # at x=0 avoids 0^(s-1) blowing up to inf when s<1, while introducing
    # no measurable integration error since tiny is indistinguishable from 0.
    # Alternative: start at x=0 and use np.nan_to_num(..., nan=0.0, posinf=0.0)
    # to suppress the inf afterward, but starting at tiny is cleaner.
    return simpsons_rule(f, s, np.finfo(float).tiny, int(1e3), int(1e5))


def factorial_gamma(x: float) -> float:
    """Return x! = Gamma(x+1), valid for any non-negative real x."""
    return np.round(euler_gamma(x + 1), 5)


display(Math(f"5! = {factorial_gamma(5):,}"))

**Comparing $\Gamma(x+1)$ to integer factorials.**
The blue curve is the continuous Gamma function evaluated across $[0, 5]$.
The red dots are the integer factorials $0!, 1!, 2!, \ldots, 5!$.
The Gamma function passes exactly through every red dot, confirming that
$\Gamma(n+1) = n!$ for all non-negative integers $n$.

In [ ]:
# Cell 03 - Plot Gamma(x+1) against integer factorials over [0, 5]

import matplotlib.pyplot as plt

xa = np.linspace(0, 5, 100)
n = [factorial_recursive(i) for i in range(6)]

plt.figure("euler_gamma_full", figsize=(8, 5))
plt.plot(xa, factorial_gamma(xa), label=r"$\Gamma(x+1)$")
plt.scatter(range(len(n)), n, color="red", zorder=5, label=r"$n!$")
plt.title("Factorial via Euler's Gamma Function")
plt.xlabel("x")
plt.ylabel("Factorial(x)")
plt.xlim(0, 5.1)
plt.legend(loc="upper left", framealpha=1.0, facecolor="white")
plt.grid(True)
plt.tight_layout()
plt.show()

**Zooming in on $[0, 2]$.**
The Gamma function has a local minimum near $x \approx 1.46$
where $\Gamma(x+1) \approx 0.886$.
This zoomed view shows clearly that the smooth continuous curve
interpolates through $0! = 1$, the minimum, and $1! = 1$, then
rises to $2! = 2$.

In [ ]:
# Cell 04 - Zoomed view over [0, 2] showing the Gamma function minimum

plt.figure("euler_gamma_zoom", figsize=(8, 5))
plt.plot(xa, factorial_gamma(xa), label=r"$\Gamma(x+1)$")
plt.scatter(range(len(n)), n, color="red", zorder=5, label=r"$n!$")
plt.title("Factorial via Euler's Gamma Function (zoomed)")
plt.xlabel("x")
plt.ylabel("Factorial(x)")
plt.xlim(0, 2.1)
plt.ylim(0.5, 2.1)
plt.legend(loc="upper left", framealpha=1.0, facecolor="white")
plt.grid(True)
plt.tight_layout()
plt.show()